In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
repos_root = Path.cwd().parent
sys.path.append(str(repos_root))

from protos import packet_pb2, ripple_pb2
from rocket_controller.encoder_decoder import PacketEncoderDecoder, DecodingNotSupportedError
import json


In [ ]:
log_dir = "/home/luanli/rocket/logs/2025_11_28_00h01m/G0T1/iteration-1"
action_file = f"{log_dir}/action-1.csv"
subscriber_file = f"{log_dir}/subscribe_event_log.csv"

df_action = pd.read_csv(action_file)
df_subscriber = pd.read_csv(subscriber_file)
df_subscriber.head()

df_TMValidation = df_action[df_action["message_type"] == "TMValidation"].copy()


def packet_data_to_dict(packet_data):
        hex_str = packet_data.strip()
        packet_bytes = bytes.fromhex(hex_str)
        pkt = packet_pb2.Packet()
        pkt.data = packet_bytes
        message, message_type = PacketEncoderDecoder.decode_packet(pkt)
        validation_dict = PacketEncoderDecoder.decode_validation(message)
        return validation_dict
    
df_TMValidation["raw_backup"] = df_TMValidation['possibly_mutated_packet_data']
df_TMValidation["possibly_mutated_packet_data"] = df_TMValidation["possibly_mutated_packet_data"].map(packet_data_to_dict)


df_TMValidation['ledger_sequence'] = df_TMValidation["possibly_mutated_packet_data"].map(
    lambda x: x.get('LedgerSequence')
)

# 2. 根据新列筛选
# df_TMValidation = df_TMValidation[df_TMValidation['ledger_sequence'] <= 15]
# 给df_TMValidation增加一列，sender：直接读取 possibly_mutated_packet_data['SigningPubKey'] 并做精确匹配到 node_info-1.csv 的 public_key 列（仅精确匹配，不做额外候选或转换）
node_info_file = log_dir + "/node_info-1.csv"
df_node_info = pd.read_csv(node_info_file)
print(df_node_info.head())

# 构造从公钥到 node_id 的精确匹配字典。这里假设 node_info 文件格式为: node_id,private_key,public_key
# 如果没有名为 'public_key' 或 'node_id' 的列，将不进行匹配（结果为 NaN）
if ('public_key' in df_node_info.columns) and ('node_id' in df_node_info.columns):
    # 去除空值并确保类型为字符串以便精确比较
    df_node_info['public_key'] = df_node_info['public_key'].astype(str).str.strip()
    df_node_info['node_id'] = df_node_info['node_id']
    pubkey_to_node = {row['public_key']: row['node_id'] for _, row in df_node_info.iterrows()}
else:
    pubkey_to_node = {}

# 直接从 possibly_mutated_packet_data 中读取 SigningPubKey 并做精确匹配
# 首先尝试直接精确匹配；如果失败且 SigningPubKey 看起来是 hex，则 deterministic 地转换为 validator base58 再匹配
import binascii

def lookup_sender_deterministic(validation_dict):
    if not isinstance(validation_dict, dict):
        return None
    spk = validation_dict.get('SigningPubKey')
    if spk is None:
        return None
    spk_s = str(spk).strip()
    # 1) 直接精确匹配
    if spk_s in pubkey_to_node:
        return pubkey_to_node[spk_s]
    # 2) 如果看起来是 hex，则转换 bytes -> xrpl validator base58 并匹配
    is_hex = False
    try:
        # hex 判断：偶数长度并且全部为十六进制数字
        if len(spk_s) % 2 == 0:
            int(spk_s, 16)
            is_hex = True
    except Exception:
        is_hex = False

    if is_hex:
        try:
            # 延迟导入 xrpl codec（如果不可用会触发异常，我们就放弃转换）
            from xrpl.core.addresscodec import codec as xrpl_codec
            raw = bytes.fromhex(spk_s)
            b58 = xrpl_codec.encode_node_public_key(raw)
            return pubkey_to_node.get(b58)
        except Exception:
            return None

    return None

# 应用映射

df_TMValidation['sender'] = df_TMValidation['possibly_mutated_packet_data'].map(lookup_sender_deterministic)
mapped = df_TMValidation['sender'].notna().sum()
total = len(df_TMValidation)
print(f'Mapped {mapped}/{total} TMValidation rows after exact match + hex->base58 conversion')

# 去重 df_TMValidation
df_TMValidation = df_TMValidation.sort_values('timestamp').drop_duplicates(
    subset=['sender', 'raw_backup'],
    keep='first'
)

print(f"去重后剩余 {len(df_TMValidation)} 行")

end_point = df_TMValidation.iloc[-1].timestamp
print(end_point)

df_action = df_action[df_action["timestamp"] <= end_point]
df_subscriber = df_subscriber[df_subscriber["timestamp"] <= end_point]
# 基本 JSON 解析（message 列是 CSV 中的 JSON 字符串）
df_subscriber['message'] = df_subscriber['message'].apply(lambda s: json.loads(s))
df_subscriber.head()


In [ ]:
nodes = sorted(df_action["from_node_id"].dropna().unique().astype(int).tolist())
node_to_y = {n: i for i, n in enumerate(nodes)}

start_time = df_action["timestamp"].min()
end_time = df_action["timestamp"].max()
sec_elapsed = (end_time - start_time) / 1000
print(f"time elapsed: {sec_elapsed:.2f} seconds, = {sec_elapsed / 60:.2f} minutes")
client_y = len(nodes)

In [ ]:
def to_sec(t):
    return (t - start_time) / 1000

In [ ]:
n_lanes = len(nodes) + 1
fig, ax = plt.subplots(figsize=(10, max(2, n_lanes * 0.5)))
ax.hlines(client_y, xmin=to_sec(start_time), xmax=to_sec(end_time), color="#eeeeee")
ax.text(to_sec(start_time) - 0.01 * (to_sec(end_time)-to_sec(start_time)), client_y, 'C', va="center", ha="right", fontsize=8)
for n, y in node_to_y.items():
    ax.hlines(y, xmin=to_sec(start_time), xmax=to_sec(end_time), color="#dddddd")
    ax.text(to_sec(start_time) - 0.01 * (to_sec(end_time)-to_sec(start_time)), y, str(n), va="center", ha="right", fontsize=8)
ax.set_ylim(-1, n_lanes)

In [ ]:
    ######### 画 validation 消息
# 创建颜色映射
# import matplotlib.cm as cm
# ledger_sequences = df_TMValidation['ledger_sequence'].dropna().unique()
# ledger_sequences = sorted([seq for seq in ledger_sequences if 1 <= seq <= 15])
# colors = cm.viridis(np.linspace(0, 1, len(ledger_sequences)))  # 或者使用其他colormap
# seq_to_color = dict(zip(ledger_sequences, colors))

# MAX_U32 = 2**32 - 1
# for _, row in df_TMValidation.iterrows():
#     try:
#         frm = int(row["from_node_id"])
#         to = int(row["to_node_id"])
#     except Exception:
#         continue
#     t = row["timestamp"]
#     delay = int(row["action"])
    
#     if delay == MAX_U32:
#         # skip drops
#         continue
    
#     seq = row["possibly_mutated_packet_data"].get("LedgerSequence", None) # 1-15
    
    

#     t2 = t + delay
#     x1 = to_sec(t)
#     x2 = to_sec(t2)
#     y1 = node_to_y.get(frm, None)
#     y2 = node_to_y.get(to, None)
#     if y1 is None or y2 is None:
#         continue
#     # color by whether delayed
#     color = seq_to_color.get(seq)
#     # draw arrow
#     ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color=color, lw=0.8))
    
# from matplotlib.patches import Patch
# legend_elements = [Patch(facecolor=seq_to_color[seq], label=f'{seq}') 
#                    for seq in ledger_sequences]
# ax.legend(handles=legend_elements, loc='best', title="LedgerSeq", fontsize=8)

# from IPython.display import display
# display(fig)


In [ ]:
# 画出每个节点的时间线：
# 一个时间轴
# 圆形：节点发送TMStatusChange，newEvent=neCLOSING_LEDGER
# 叉x：节点发送TMStatusChange，newEvent=neACCEPTED_LEDGER
# 竖直向下箭头：向订阅者发送ledgerClosed消息
# 正方形：节点收到validation的消息
# 斜向右上45度箭头：节点发送validation消息
# 不同类型事件用不同颜色，在事件旁标注seq号

import json

def parse_status_change(original_data: str):
    parts = original_data.split(';')
    res = {}
    for part in parts:
        kv = part.split(':')
        if len(kv) == 2:
            key, value = kv
            res[key.strip()] = value.strip()
    return res
def get_node_timeline(node_id):
    events = {} # sec_elapsed -> (event, seq, sender)
    seen_events = set()  # 用于记录已经出现的事件组合
    
    # status change
    for _, row in df_action.iterrows():
        sec = (row["timestamp"] - start_time) / 1000
        if int(row["from_node_id"]) != node_id:
            continue
        if row["message_type"] == "TMStatusChange":
            msg = parse_status_change(row["original_data"])
            if msg["newEvent"] in ["neCLOSING_LEDGER", "neACCEPTED_LEDGER"]:
                event_key = (msg["newEvent"], msg["ledgerSeq"], row["from_node_id"])
                if event_key not in seen_events:
                    events[sec] = (msg["newEvent"], msg["ledgerSeq"], row["from_node_id"])
                    seen_events.add(event_key)

    # ledger closed
    for _, row in df_subscriber.iterrows():
        sec = (row["timestamp"] - start_time) / 1000
        if int(row["node_id"]) != node_id:
            continue
        if row["message"].get("type") == "ledgerClosed":
            ledger_seq = row["message"].get("ledger_index")
            event_key = ("ledgerClosed", ledger_seq, row["node_id"])
            if event_key not in seen_events:
                events[sec] = ("ledgerClosed", ledger_seq, row["node_id"])
                seen_events.add(event_key)
            
    # validation received and sent
    for _, row in df_TMValidation.iterrows():
        sec = (row["timestamp"] - start_time) / 1000
        if int(row["to_node_id"]) == node_id:
            ledger_seq = row["possibly_mutated_packet_data"].get("LedgerSequence")
            event_key = ("validationReceived", ledger_seq, row["sender"])
            if event_key not in seen_events:
                events[sec] = ("validationReceived", ledger_seq, row["sender"])
                seen_events.add(event_key)
        elif int(row["sender"]) == node_id:
            ledger_seq = row["possibly_mutated_packet_data"].get("LedgerSequence")
            event_key = ("validationSent", ledger_seq, row["sender"])
            if event_key not in seen_events:
                events[sec] = ("validationSent", ledger_seq, row["sender"])
                seen_events.add(event_key)
    
    return events

events = get_node_timeline(1)
events

In [ ]:
import matplotlib.pyplot as plt


def plot_node_timeline(node_id, events):
    if not events:
        print(f"Node {node_id} has no events")
        return

    # 创建图形
    fig, ax = plt.subplots(figsize=(15, 8))

    # 定义事件类型到颜色的映射
    event_config = {
        "neCLOSING_LEDGER": {"color": "blue", "label": "CLOSING_LEDGER"},
        "neACCEPTED_LEDGER": {"color": "red", "label": "ACCEPTED_LEDGER"},
        "ledgerClosed": {"color": "green", "label": "ledgerClosed"},
        "validationReceived": {"color": "orange", "label": "validationReceived"},
        "validationSent": {"color": "purple", "label": "validationSent"},
    }

    # 设置y轴位置（所有事件都在同一水平线上）
    y_pos = 0

    # 绘制时间轴
    timestamps = [ts for ts, *_ in events.items()]
    ax.axhline(y=y_pos, color="black", linewidth=1, alpha=0.5)

    YLIM = 1

    # 绘制每个事件
    for timestamp, (event_type, *seq) in events.items():
        config = event_config.get(event_type, {"color": "black", "label": event_type})

        # 所有箭头都垂直，但方向不同
        if event_type == "validationSent":
            # validationSent：箭头向外（从时间轴向上）
            ax.annotate(
                "",
                xy=(timestamp, y_pos + 0.3 * YLIM),  # 箭头终点向上
                xytext=(timestamp, y_pos),  # 箭头起点在时间轴
                arrowprops=dict(
                    arrowstyle="->", color=config["color"], lw=1, alpha=0.8
                ),
            )
        elif event_type == "validationReceived":
            # validationReceived：在时间轴下方，所以应该向上指（向时间轴）
            ax.annotate(
                "",
                xy=(timestamp, y_pos),  # 箭头终点在时间轴
                xytext=(timestamp, y_pos - 0.3 * YLIM),  # 箭头起点在下方
                arrowprops=dict(
                    arrowstyle="->", color=config["color"], lw=1, alpha=0.8
                ),
            )
        elif event_type in [
            "neCLOSING_LEDGER",
            "neACCEPTED_LEDGER",
        ]:
            ax.annotate(
                "",
                xy=(timestamp, y_pos + 0.5 * YLIM),  # 箭头终点向上
                xytext=(timestamp, y_pos),  # 箭头起点在时间轴
                arrowprops=dict(
                    arrowstyle="->", color=config["color"], lw=1, alpha=0.8
                ),
            )
        elif event_type == "ledgerClosed":
            # 在时间轴下方，但是要向下指
            ax.annotate(
                "",
                xy=(timestamp, y_pos - 0.5 * YLIM),  # 箭头终点向下
                xytext=(timestamp, y_pos),  # 箭头起点在时间轴
                arrowprops=dict(
                    arrowstyle="->", color=config["color"], lw=1, alpha=0.8
                ),
            )


        print(config)

        # # 添加seq号标注
        # ax.annotate(f'{seq}', xy=(timestamp, y_pos), xytext=(5, 5),
        #            textcoords='offset points', fontsize=8, alpha=0.7)

    # 设置图形属性
    ax.set_title(f"Node {node_id} Timeline", fontsize=14, fontweight="bold")
    ax.set_xlabel("Time (s)", fontsize=12)
    ax.set_ylabel("Events", fontsize=12)

    # 隐藏y轴刻度
    ax.set_yticks([])

    # 添加图例（颜色图例）
    legend_elements = []
    for event_type, config in event_config.items():
        legend_elements.append(
            plt.Line2D([0], [0], color=config["color"], lw=3, label=config["label"])
        )
    ax.legend(handles=legend_elements, loc="upper right")

    # 调整x轴范围，留一些边距
    min_ts = min(timestamps)
    max_ts = max(timestamps)
    margin = (max_ts - min_ts) * 0.05
    ax.set_xlim(min_ts - margin, max_ts + margin)

    # 调整y轴范围，为箭头留出空间
    ax.set_ylim(-YLIM, YLIM)

    plt.tight_layout()
    out_dir = Path("out")
    out_dir.mkdir(exist_ok=True)
    out_file = out_dir / f"node_{node_id}_timeline.pdf"
    plt.savefig(out_file)
    plt.show()


# 使用示例
for node_id in nodes:
    events = get_node_timeline(node_id)
    plot_node_timeline(node_id, events)